In [1]:
import pandas as pd
import looker_sdk
from io import StringIO

In [2]:
# Load the Looker Python SDK tools.
import looker_sdk

# Path to the `looker.ini` file we setup.


CREDENTIALS_PATH = "utils\\looker.ini"  # <--- Change


# Initializing the Looker SDK.


sdk = looker_sdk.init40(CREDENTIALS_PATH)

# Retrieve my user information from Looker.
my_user = sdk.me()

# Display my username.
my_user["display_name"]

'Josh Allen'

In [3]:
#############
# Parameters
#############
RESULT_FORMAT = "csv"

#############
# Send Query
#############

# Define your query
body = {
    "model": "oahu_connection_model",
    "view": "base",
    "fields": [
        "programs.id",
        "enrollments.reporting_period_start_date",
        "enrollments.reporting_period_end_date",
        "program_inventory.start_date",
        "program_inventory.end_date",
        "program_inventory.total_bed_inventory",
    ],
    "filters": {
        "enrollments.date_filter": "this quarter",
    },
    "limit": 100000,
}

################
# Get Response
################

# Run the inline query
result = sdk.run_inline_query(result_format=RESULT_FORMAT, body=body)

# Convert to dataframe.
df = pd.read_csv(StringIO(result))

In [4]:
df

,Programs Program ID,Enrollments Reporting Period Start Date,Enrollments Reporting Period End Date,Bed and Unit Inventory Start Date,Bed and Unit Inventory End Date,Bed and Unit Inventory Total Bed Inventory
0,18,2024-10-01,2025-01-01,NaN,NaN,0
1,20,2024-10-01,2025-01-01,2002-05-15,2024-01-21,153
2,20,2024-10-01,2025-01-01,2024-01-22,NaN,154
3,33,2024-10-01,2025-01-01,2020-06-15,2024-01-21,134
4,33,2024-10-01,2025-01-01,2023-01-23,2023-10-26,134
...,...,...,...,...,...,...
185,353,2024-10-01,2025-01-01,2024-02-01,NaN,19
186,354,2024-10-01,2025-01-01,2024-04-15,NaN,40
187,357,2024-10-01,2025-01-01,2024-03-01,NaN,12
188,358,2024-10-01,2025-01-01,NaN,NaN,0


In [5]:
from datetime import datetime

In [6]:
df["Enrollments Reporting Period Start Date"] = pd.to_datetime(
    df["Enrollments Reporting Period Start Date"]
).copy()
df["Enrollments Reporting Period End Date"] = pd.to_datetime(
    df["Enrollments Reporting Period End Date"]
).copy()
df["Bed and Unit Inventory Start Date"] = pd.to_datetime(
    df["Bed and Unit Inventory Start Date"]
).copy()
df["Bed and Unit Inventory End Date"] = pd.to_datetime(
    df["Bed and Unit Inventory End Date"]
).copy()
today = pd.Timestamp(datetime.now().date())
df["Bed and Unit Inventory End Date"] = (
    df["Bed and Unit Inventory End Date"].fillna(today).copy()
)
df = df[~df["Bed and Unit Inventory Start Date"].isna()].copy()
df["overlap_days"] = (
    pd.to_datetime(
        df[
            ["Enrollments Reporting Period End Date", "Bed and Unit Inventory End Date"]
        ].min(axis=1)
    )
    - pd.to_datetime(
        df[
            [
                "Enrollments Reporting Period Start Date",
                "Bed and Unit Inventory Start Date",
            ]
        ].max(axis=1)
    )
).dt.days + 1

# Ensure non-negative overlaps (if no overlap, set to 0)
df["overlap_days"] = df["overlap_days"].apply(lambda x: max(0, x))
df["Bed and Unit Inventory Total Bed Inventory for Reporting Period"] = (
    df["Bed and Unit Inventory Total Bed Inventory"] * df["overlap_days"]
)
df.pivot_table(
    index="Programs Program ID",
    values="Bed and Unit Inventory Total Bed Inventory for Reporting Period",
    aggfunc="sum",
).reset_index(drop=False)